<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Vector%20Database%20Mini%20Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu sentence-transformers

import faiss
import numpy as np
import time
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 31.7 MB/s  0:00:00


In [4]:
#1.SELECT A DATASET(500+ items)
#Generating a synthetic dataset of 1000 tech/science document snippets
topics = ["artificial intelligence","climate change","quantum computing","renewable energy","space exploration"]
dataset = [f"This is a research document focusing on {topic} and its impact on future technologies{i}."
           for i in range (1000) for topic in topics]

print(f"Dataset created with {len(dataset)}.items")

Dataset created with 5000.items


In [5]:
#2.GENERATE EMBEDDINGS
print(f"Loading embedding model and vectors")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(dataset,show_progress_bar=True)
dimension=embeddings.shape[1]
print(f"Embeddings generated.Dimension:{dimension}")

Loading embedding model and vectors


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings generated.Dimension:384


In [7]:
#3.CHOOSE INDEXING METHOD AND BUILD INDEXES
#Exact search(Flat L2)
print("Building Exact Index(IndexFlatL2)..")
index_exact=faiss.IndexFlatL2(dimension)
index_exact.add(embeddings)

#Approximate search (IVF - Inverted File Index)
print("Building Approximate Index(IndexIVFFlat)...")
nlist = 20 # Number of clusters
quantizer = faiss.IndexFlatL2(dimension)
index_approx = faiss.IndexIVFFlat(quantizer,dimension,nlist,faiss.METRIC_L2)
index_approx.train(embeddings) # Train the index to find cluster centroids
index_approx.add(embeddings)
index_approx.nprobe = 5  # Number of clusters to search during query


Building Exact Index(IndexFlatL2)..
Building Approximate Index(IndexIVFFlat)...


In [10]:
#4.IMPLEMENT SEARCH PIPELINE AND EVALUATE
query = "Tell me about the future of machine learning and AI"
query_vec = model.encode([query])
k = 5 # Top k similar items

#Run Exact Search
start_time = time.time()
D_Exact,I_Exact = index_exact.search(query_vec,k)
time_exact = time.time() - start_time

#Run Approximate Search
start_time = time.time()
D_Approx,I_Approx = index_approx.search(query_vec,k)
time_approx = time.time() - start_time

#Calculate Recall@k
#Recall@k = (Number of common items in both results)/k
common_items = len(set(I_Exact[0]).intersection(set(I_Approx[0])))
recall_at_k = common_items/k

print("\nSEARCH RESULTS & METRICS---")
print(f"Query: '{query}'")
print(f"Top {k} Exact Results: {[dataset[i] for i in I_Exact[0]]}")
print(f"Top {k} Approx Results: {[dataset[i] for i in I_Approx[0]]}")
print(f"\nQuery Time(Exact): {time_exact:.6f} seconds")
print(f"Query Time(Approx): {time_approx:.6f} seconds")
print(f"Recall@{k}: {recall_at_k:.2f} (1.00 means perfect match with exact search)")


SEARCH RESULTS & METRICS---
Query: 'Tell me about the future of machine learning and AI'
Top 5 Exact Results: ['This is a research document focusing on artificial intelligence and its impact on future technologies149.', 'This is a research document focusing on artificial intelligence and its impact on future technologies146.', 'This is a research document focusing on artificial intelligence and its impact on future technologies914.', 'This is a research document focusing on artificial intelligence and its impact on future technologies14.', 'This is a research document focusing on artificial intelligence and its impact on future technologies712.']
Top 5 Approx Results: ['This is a research document focusing on artificial intelligence and its impact on future technologies149.', 'This is a research document focusing on artificial intelligence and its impact on future technologies146.', 'This is a research document focusing on artificial intelligence and its impact on future technologies9

In [12]:
# 5. DOCUMENT OPTIMIZATION STRATEGIES
print("\n--- OPTIMIZATION STRATEGY: SCALAR QUANTIZATION ---")
# Implementing SQ8 (Scalar Quantization) to reduce memory footprint
index_sq = faiss.IndexIVFScalarQuantizer(quantizer, dimension, nlist, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_L2)
index_sq.train(embeddings)
index_sq.add(embeddings)
index_sq.nprobe = 5

# Check memory usage (approximate)
mem_exact = index_exact.ntotal * dimension * 4 / (1024 * 1024) # 4 bytes per float32
mem_sq = index_sq.ntotal * dimension * 1 / (1024 * 1024)      # 1 byte per uint8
print(f"Memory usage Exact (Flat32): {mem_exact:.2f} MB")
print(f"Memory usage Optimized (SQ8): {mem_sq:.2f} MB")


--- OPTIMIZATION STRATEGY: SCALAR QUANTIZATION ---
Memory usage Exact (Flat32): 7.32 MB
Memory usage Optimized (SQ8): 1.83 MB
